In [ ]:
import pandas as pd
import seaborn as sns
import numpy as np

In [5]:
prep_crash_df = pd.read_csv(
    "https://raw.githubusercontent.com/Alissa-Ouspen/data201_alissa/main/final/drivers_data_final_df.csv")

In [ ]:
prep_crash_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 214273 entries, 0 to 214272
Data columns (total 30 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   Vehicle_Num                 214273 non-null  int64  
 1   Agency_Name                 214273 non-null  object 
 2   ACRS_Report_Type            214273 non-null  object 
 3   Crash_Date_Time             214273 non-null  object 
 4   Related_Non_Motorist        6767 non-null    object 
 5   Collision_Type              191162 non-null  object 
 6   Weather                     199566 non-null  object 
 7   Surface_Condition           189141 non-null  object 
 8   Ambient_Light               211494 non-null  object 
 9   Traffic_Control             182332 non-null  object 
 10  Driver_Substance_Use        170904 non-null  object 
 11  Non_Motorist_Substance_Use  5654 non-null    object 
 12  Driver_At_Fault             209591 non-null  object 
 13  Injury_Severit

In [6]:
prep_crash_df.nunique()

,0
Vehicle_Num,214273
Agency_Name,6
ACRS_Report_Type,3
Crash_Date_Time,117933
Related_Non_Motorist,25
Collision_Type,6
Weather,15
Surface_Condition,11
Ambient_Light,9
Traffic_Control,28


## Data Cleaning and Feature Engineering

In [9]:
prep_crash_df["Crash_Date_Time"] = pd.to_datetime(prep_crash_df["Crash_Date_Time"])
prep_crash_df["Crash_Hour"] = prep_crash_df["Crash_Date_Time"].dt.hour
prep_crash_df["Crash_Month"] = prep_crash_df["Crash_Date_Time"].dt.month
prep_crash_df["Crash_Year"] = prep_crash_df["Crash_Date_Time"].dt.year

display(prep_crash_df.head())

,Vehicle_Num,Agency_Name,ACRS_Report_Type,Crash_Date_Time,Related_Non_Motorist,Collision_Type,Weather,Surface_Condition,Ambient_Light,Traffic_Control,...,Lane_Type,At_Fault,First_Harmful_Event,Junction,Intersection_Type,Road_Alignment,Road_Condition,Crash_Hour,Crash_Month,Crash_Year
0,0,Montgomery County Police,Injury Crash,2026-05-05 22:43:00,NaN,HEAD ON COLLISIONS,Clear,Dry,Dark - Lighted,No Controls,...,Lane 2,DRIVER,MOTOR VEHICLE IN TRANSPORT,NaN,NaN,Straight,No Defects,22,5,2026
1,1,Montgomery County Police,Injury Crash,2026-05-05 22:43:00,NaN,HEAD ON COLLISIONS,Clear,Dry,Dark - Lighted,No Controls,...,Lane 2,DRIVER,MOTOR VEHICLE IN TRANSPORT,NaN,NaN,Straight,No Defects,22,5,2026
2,2,Rockville Police Dept,Property Damage Crash,2026-05-05 21:43:00,NaN,REAR END COLLISIONS,Clear,Dry,Dark - Lighted,Flashing Traffic Control Signal,...,Lane 2,NaN,MOTOR VEHICLE IN TRANSPORT,ACCELERATION/DECELERATION LANE,NaN,Straight,No Defects,21,5,2026
3,3,Rockville Police Dept,Property Damage Crash,2026-05-05 21:43:00,NaN,REAR END COLLISIONS,Clear,Dry,Dark - Lighted,Flashing Traffic Control Signal,...,Lane 2,NaN,MOTOR VEHICLE IN TRANSPORT,ACCELERATION/DECELERATION LANE,NaN,Straight,No Defects,21,5,2026
4,4,Montgomery County Police,Property Damage Crash,2026-05-05 21:30:00,NaN,ANGLE COLLISIONS,Clear,Dry,Dark - Lighted,Lane Use Control Signal,...,Lane 1,DRIVER,MOTOR VEHICLE IN TRANSPORT,CROSSOVER RELATED,NaN,Straight,No Defects,21,5,2026


In [12]:
display(prep_crash_df["Driver_Substance_Use"].value_counts())

,count
Driver_Substance_Use,
NONE DETECTED,122544
"Not Suspect of Alcohol Use, Not Suspect of Drug Use",36007
"Unknown, Unknown",4720
ALCOHOL PRESENT,4087
ALCOHOL CONTRIBUTED,1435
"Suspect of Alcohol Use, Not Suspect of Drug Use",1013
ILLEGAL DRUG PRESENT,259
"Suspect of Alcohol Use, Unknown",126
MEDICATION PRESENT,117


| Driver_Substance_Use | alcohol_use | illegal_drug_use | medication_use |
| :------------------- | :------------- | :------------- | :------------- |
| Not Suspect of Alcohol Use, Not Suspect of Drug Use |No |No | |
| Suspect of Alcohol Use, Not Suspect of Drug Use | Yes|No | |
| Unknown, Unknown | | | |
| Suspect of Alcohol Use, Unknown |Yes | | |
| Not Suspect of Alcohol Use, Suspect of Drug Use | No|Yes | |
| Suspect of Alcohol Use, Suspect of Drug Use |Yes |Yes | |
| Not Suspect of Alcohol Use, Unknown |No | | |
| Unknown, Not Suspect of Drug Use | |No | |
| Unknown, Suspect of Drug Use | |Yes | |
| NONE DETECTED | | | |
| NaN | | | |
| ALCOHOL PRESENT | Yes| | |
| ALCOHOL CONTRIBUTED | Yes| | |
| ILLEGAL DRUG PRESENT | | Yes| |
| MEDICATION CONTRIBUTED | | |Yes |
| COMBINATION CONTRIBUTED | Yes|Yes | |
| ILLEGAL DRUG CONTRIBUTED | | Yes| |
| COMBINED SUBSTANCE PRESENT | Yes| Yes| |
| MEDICATION PRESENT | | | Yes|

In [11]:
print(prep_crash_df["Driver_Substance_Use"].unique())

['Not Suspect of Alcohol Use, Not Suspect of Drug Use'
 'Suspect of Alcohol Use, Not Suspect of Drug Use' 'Unknown, Unknown'
 'Suspect of Alcohol Use, Unknown'
 'Not Suspect of Alcohol Use, Suspect of Drug Use'
 'Suspect of Alcohol Use, Suspect of Drug Use'
 'Not Suspect of Alcohol Use, Unknown' 'Unknown, Not Suspect of Drug Use'
 'Unknown, Suspect of Drug Use' 'NONE DETECTED' nan 'ALCOHOL PRESENT'
 'ALCOHOL CONTRIBUTED' 'ILLEGAL DRUG PRESENT' 'MEDICATION CONTRIBUTED'
 'COMBINATION CONTRIBUTED' 'ILLEGAL DRUG CONTRIBUTED'
 'COMBINED SUBSTANCE PRESENT' 'MEDICATION PRESENT']


In [19]:
def get_alcohol_use_from_table(text):
    if pd.isna(text) or text == "Unknown, Unknown" or "unknown" in str(text).lower() or text == "NONE DETECTED":
        return "NaN"
    text_lower = text.lower()
    if text_lower == "not suspect of alcohol use, not suspect of drug use" or \
       text_lower == "not suspect of alcohol use, suspect of drug use" or \
       text_lower == "not suspect of alcohol use, unknown":
        return "No"
    elif text_lower == "suspect of alcohol use, not suspect of drug use" or \
         text_lower == "suspect of alcohol use, unknown" or \
         text_lower == "suspect of alcohol use, suspect of drug use" or \
         text_lower == "alcohol present" or \
         text_lower == "alcohol contributed" or \
         text_lower == "combination contributed" or \
         text_lower == "combined substance present":
        return "Yes"
    return "NaN"

def get_illegal_drug_use_from_table(text):
    if pd.isna(text) or text == "Unknown, Unknown" or "unknown" in str(text).lower() or text == "NONE DETECTED":
        return "NaN"
    text_lower = text.lower()
    if text_lower == "not suspect of alcohol use, not suspect of drug use" or \
       text_lower == "suspect of alcohol use, not suspect of drug use" or \
       text_lower == "unknown, not suspect of drug use":
        return "No"
    elif text_lower == "not suspect of alcohol use, suspect of drug use" or \
         text_lower == "suspect of alcohol use, suspect of drug use" or \
         text_lower == "unknown, suspect of drug use" or \
         text_lower == "illegal drug present" or \
         text_lower == "combination contributed" or \
         text_lower == "illegal drug contributed" or \
         text_lower == "combined substance present":
        return "Yes"
    return "NaN"

def get_medication_use_from_table(text):
    if pd.isna(text) or text == "Unknown, Unknown" or "unknown" in str(text).lower() or text == "NONE DETECTED":
        return "NaN"
    text_lower = text.lower()
    if text_lower == "medication contributed" or \
       text_lower == "medication present":
        return "Yes"
    return "NaN"

prep_crash_df["alcohol_use"] = prep_crash_df["Driver_Substance_Use"].apply(get_alcohol_use_from_table)
prep_crash_df["illegal_drug_use"] = prep_crash_df["Driver_Substance_Use"].apply(get_illegal_drug_use_from_table)
prep_crash_df["medication_use"] = prep_crash_df["Driver_Substance_Use"].apply(get_medication_use_from_table)

print("Value counts for 'alcohol_use':")
display(prep_crash_df["alcohol_use"].value_counts(dropna=False))
print("\nValue counts for 'illegal_drug_use':")
display(prep_crash_df["illegal_drug_use"].value_counts(dropna=False))
print("\nValue counts for 'medication_use':")
display(prep_crash_df["medication_use"].value_counts(dropna=False))

Value counts for 'alcohol_use':


,count
alcohol_use,
NaN,171463
No,36058
Yes,6752



Value counts for 'illegal_drug_use':


,count
illegal_drug_use,
NaN,176624
No,37020
Yes,629



Value counts for 'medication_use':


,count
medication_use,
NaN,214092
Yes,181


In [15]:
display(prep_crash_df.head())

,Vehicle_Num,Agency_Name,ACRS_Report_Type,Crash_Date_Time,Related_Non_Motorist,Collision_Type,Weather,Surface_Condition,Ambient_Light,Traffic_Control,...,Junction,Intersection_Type,Road_Alignment,Road_Condition,Crash_Hour,Crash_Month,Crash_Year,alcohol_use,illegal_drug_use,medication_use
0,0,Montgomery County Police,Injury Crash,2026-05-05 22:43:00,NaN,HEAD ON COLLISIONS,Clear,Dry,Dark - Lighted,No Controls,...,NaN,NaN,Straight,No Defects,22,5,2026,No,No,NaN
1,1,Montgomery County Police,Injury Crash,2026-05-05 22:43:00,NaN,HEAD ON COLLISIONS,Clear,Dry,Dark - Lighted,No Controls,...,NaN,NaN,Straight,No Defects,22,5,2026,Yes,No,NaN
2,2,Rockville Police Dept,Property Damage Crash,2026-05-05 21:43:00,NaN,REAR END COLLISIONS,Clear,Dry,Dark - Lighted,Flashing Traffic Control Signal,...,ACCELERATION/DECELERATION LANE,NaN,Straight,No Defects,21,5,2026,No,No,NaN
3,3,Rockville Police Dept,Property Damage Crash,2026-05-05 21:43:00,NaN,REAR END COLLISIONS,Clear,Dry,Dark - Lighted,Flashing Traffic Control Signal,...,ACCELERATION/DECELERATION LANE,NaN,Straight,No Defects,21,5,2026,Yes,No,NaN
4,4,Montgomery County Police,Property Damage Crash,2026-05-05 21:30:00,NaN,ANGLE COLLISIONS,Clear,Dry,Dark - Lighted,Lane Use Control Signal,...,CROSSOVER RELATED,NaN,Straight,No Defects,21,5,2026,No,No,NaN


In [21]:
prep_crash_df["Weather"] = prep_crash_df["Weather"].replace("RAINING", "Rain")
prep_crash_df["Weather"] = prep_crash_df["Weather"].replace([
    "Snow", "WINTRY MIX", "SLEET", "Blowing Snow", "Sleet Or Hail"
], "Snow/ Sleet/ Hail")
prep_crash_df["Weather"] = prep_crash_df["Weather"].replace("FOGGY", "Fog, Smog, Smoke")
display(prep_crash_df["Weather"].value_counts(dropna=False))

,count
Weather,
Clear,149986
Rain,24774
Cloudy,21011
NaN,14707
Snow/ Sleet/ Hail,2735
"Fog, Smog, Smoke",775
SEVERE WINDS,154
Freezing Rain Or Freezing Drizzle,79
Severe Crosswinds,36


In [22]:
display(prep_crash_df["Ambient_Light"].value_counts(dropna=False))

,count
Ambient_Light,
Daylight,145384
DARK LIGHTS ON,39551
Dark - Lighted,9874
DARK NO LIGHTS,4966
Dusk,4525
Dawn,3899
NaN,2779
DARK -- UNKNOWN LIGHTING,1578
Dark - Not Lighted,1468


In [24]:
prep_crash_df["Ambient_Light"] = prep_crash_df["Ambient_Light"].replace("DARK LIGHTS ON", "Dark - Lighted")
prep_crash_df["Ambient_Light"] = prep_crash_df["Ambient_Light"].replace(
    ["DARK NO LIGHTS", "DARK -- UNKNOWN LIGHTING", "Dark - Not Lighted", "Dark - Unknown Lighting"], "Dark"
)
display(prep_crash_df["Ambient_Light"].value_counts(dropna=False))

,count
Ambient_Light,
Daylight,145384
Dark - Lighted,49425
Dark,8261
Dusk,4525
Dawn,3899
NaN,2779


In [25]:
prep_crash_df["Vehicle_Age"] = prep_crash_df["Crash_Year"] - prep_crash_df["Vehicle_Year"]

# Identify rows where Vehicle_Age is negative
negative_age_mask = prep_crash_df["Vehicle_Age"] < 0

# Set 'Vehicle_Year' to NaN for these rows
prep_crash_df.loc[negative_age_mask, "Vehicle_Year"] = np.nan

# Recalculate 'Vehicle_Age' to reflect the NaN changes in 'Vehicle_Year'
prep_crash_df["Vehicle_Age"] = prep_crash_df["Crash_Year"] - prep_crash_df["Vehicle_Year"]

print("Value counts for 'Vehicle_Age':")
display(prep_crash_df["Vehicle_Age"].value_counts(dropna=False))

print("\nFirst 5 rows with new 'Vehicle_Age' column:")
display(prep_crash_df[['Crash_Year', 'Vehicle_Year', 'Vehicle_Age']].head())

Value counts for 'Vehicle_Age':


,count
Vehicle_Age,
4.0,14222
3.0,13889
5.0,13740
1.0,13730
2.0,13654
...,...
51.0,1
54.0,1
64.0,1



First 5 rows with new 'Vehicle_Age' column:


,Crash_Year,Vehicle_Year,Vehicle_Age
0,2026,2018.0,8.0
1,2026,2026.0,0.0
2,2026,2023.0,3.0
3,2026,2005.0,21.0
4,2026,2025.0,1.0


In [26]:
#  save cleaned df to csv to back up

prep_crash_df.to_csv("drivers_final2_df.csv", index=False)

# index = False prevents "Unnamed: 0" column from being added each time file loads.

Ignore:

AI use tracker:  
 - From the "Driver_Substance_Use" field, in prep_crash_df create three new columns:  
 "alcohol_use" , "illegal_drug_use", "medication_use".   
 Follow the markdown chart I have filled out.  If a cell is left blank, fill with NaN

- Create another colomn: "Vehicle_Age". Fill with the difference between "Crash_Year" and "Vehicle_Year".  
Meaning, subtract vehicle_year from crash_year.  
If the difference results in a negative value, then change the "Vehicle_Year" value for that cell with NaN
